# Beginner 04 — Authorization for Agents

## Enterprise scenario

A customer-service **refund agent** acts on behalf of employees and customers. It can inspect orders and propose refunds, but authority depends on the actor, requester, resource, amount, task, delegation, workload, and approval state.

We implement authorization models directly so their semantics are visible before later courses introduce OPA, Cedar, and OpenFGA.

### Lab outcomes

- implement default deny;
- compare RBAC, ABAC, ReBAC, and capabilities;
- separate PEP and PDP;
- create task-scoped delegation;
- prevent privilege amplification;
- enforce separation of duties;
- filter RAG data before model exposure;
- generate decision evidence;
- run adversarial authorization tests.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any, Optional
import json, uuid

def now():
    return datetime.now(timezone.utc)


## 1 — Authorization request

In [ ]:
@dataclass(frozen=True)
class AuthzRequest:
    requester: str
    actor: str
    workload: str
    action: str
    resource: str
    context: dict[str, Any] = field(default_factory=dict)

req = AuthzRequest(
    requester="user:alice",
    actor="agent:refund-specialist",
    workload="spiffe://corp.example/prod/refund",
    action="refund",
    resource="order:123",
    context={
        "amount": 120,
        "tenant": "north",
        "task_id": "task:928",
        "risk": "low",
    },
)
req


## 2 — Default deny

In [ ]:
@dataclass(frozen=True)
class Decision:
    allowed: bool
    reason: str
    model: str

def default_deny() -> Decision:
    return Decision(False, "No explicit authorization matched", "default")

print(default_deny())


## 3 — RBAC

In [ ]:
ROLE_ASSIGNMENTS = {
    "agent:refund-specialist": {"refund_agent"},
    "user:alice": {"customer_service"},
}

ROLE_PERMISSIONS = {
    "refund_agent": {"order:read", "refund:create"},
    "customer_service": {"order:read"},
}

ACTION_PERMISSION = {
    "read_order": "order:read",
    "refund": "refund:create",
    "delete_order": "order:delete",
}

def authorize_rbac(request: AuthzRequest) -> Decision:
    needed = ACTION_PERMISSION.get(request.action)
    if needed is None:
        return Decision(False, "Unknown action", "RBAC")

    for role in ROLE_ASSIGNMENTS.get(request.actor, set()):
        if needed in ROLE_PERMISSIONS.get(role, set()):
            return Decision(True, f"Role {role} grants {needed}", "RBAC")

    return Decision(False, f"No role grants {needed}", "RBAC")

for action in ["read_order", "refund", "delete_order"]:
    print(action, "->", authorize_rbac(AuthzRequest(
        req.requester, req.actor, req.workload, action, req.resource, req.context
    )))


### RBAC limitation

The role grants `refund:create`, but it cannot naturally express:

```text
only order:123
only <= CAD 200
only for task:928
only for Alice
```

You can add more roles, but task/resource combinations quickly create role explosion.


## 4 — ABAC

In [ ]:
ACTOR_ATTRIBUTES = {
    "agent:refund-specialist": {
        "type": "agent",
        "department": "customer-service",
        "environment": "prod",
        "risk_tier": "medium",
    }
}

RESOURCE_ATTRIBUTES = {
    "order:123": {"tenant": "north", "status": "paid"},
    "order:999": {"tenant": "south", "status": "paid"},
}

REQUESTER_ATTRIBUTES = {
    "user:alice": {"tenant": "north", "department": "customer-service"},
}

def authorize_abac(request: AuthzRequest) -> Decision:
    actor = ACTOR_ATTRIBUTES.get(request.actor, {})
    resource = RESOURCE_ATTRIBUTES.get(request.resource, {})
    requester = REQUESTER_ATTRIBUTES.get(request.requester, {})

    if request.action != "refund":
        return Decision(False, "ABAC policy applies only to refund", "ABAC")
    if actor.get("type") != "agent":
        return Decision(False, "Actor is not an agent", "ABAC")
    if actor.get("department") != "customer-service":
        return Decision(False, "Wrong department", "ABAC")
    if actor.get("environment") != "prod":
        return Decision(False, "Non-production actor", "ABAC")
    if resource.get("tenant") != requester.get("tenant"):
        return Decision(False, "Tenant mismatch", "ABAC")
    if request.context.get("amount", float("inf")) > 200:
        return Decision(False, "Refund exceeds CAD 200", "ABAC")
    if request.context.get("risk") != "low":
        return Decision(False, "Risk is not low", "ABAC")

    return Decision(True, "All ABAC conditions satisfied", "ABAC")

print(authorize_abac(req))
print(authorize_abac(AuthzRequest(
    req.requester, req.actor, req.workload, "refund", "order:999", req.context
)))


## 5 — ReBAC relationship graph

In [ ]:
RELATIONS = {
    ("user:alice", "member", "tenant:north"),
    ("order:123", "belongs_to", "tenant:north"),
    ("order:999", "belongs_to", "tenant:south"),
    ("agent:refund-specialist", "can_act_on_behalf_of", "user:alice"),
    ("agent:refund-specialist", "assigned", "task:928"),
    ("task:928", "targets", "order:123"),
}

def related(source, relation, target):
    return (source, relation, target) in RELATIONS

def authorize_rebac(request: AuthzRequest) -> Decision:
    task = request.context.get("task_id")
    if request.action != "refund":
        return Decision(False, "Only refund relation modeled", "ReBAC")
    if not related(request.actor, "can_act_on_behalf_of", request.requester):
        return Decision(False, "No actor/requester delegation relationship", "ReBAC")
    if not related(request.actor, "assigned", task):
        return Decision(False, "Agent not assigned to task", "ReBAC")
    if not related(task, "targets", request.resource):
        return Decision(False, "Task does not target resource", "ReBAC")
    return Decision(True, "Required relationships exist", "ReBAC")

print(authorize_rebac(req))


This graph is deliberately tiny. Production ReBAC engines such as OpenFGA evaluate relationship tuples and inferred permissions at scale.

## 6 — Capability-style task grant

In [ ]:
@dataclass(frozen=True)
class Capability:
    id: str
    holder: str
    action: str
    resource: str
    max_amount: float
    expires_at: datetime
    task_id: str

capability = Capability(
    id=f"cap:{uuid.uuid4().hex[:8]}",
    holder="agent:refund-specialist",
    action="refund",
    resource="order:123",
    max_amount=200,
    expires_at=now() + timedelta(minutes=15),
    task_id="task:928",
)

def authorize_capability(request: AuthzRequest, cap: Capability) -> Decision:
    if cap.holder != request.actor:
        return Decision(False, "Capability holder mismatch", "Capability")
    if cap.action != request.action:
        return Decision(False, "Action outside capability", "Capability")
    if cap.resource != request.resource:
        return Decision(False, "Resource outside capability", "Capability")
    if cap.task_id != request.context.get("task_id"):
        return Decision(False, "Task mismatch", "Capability")
    if request.context.get("amount", float("inf")) > cap.max_amount:
        return Decision(False, "Amount exceeds capability", "Capability")
    if now() >= cap.expires_at:
        return Decision(False, "Capability expired", "Capability")
    return Decision(True, "Valid bounded capability", "Capability")

print(authorize_capability(req, capability))


## 7 — Compare models on the same request

In [ ]:
for fn in [authorize_rbac, authorize_abac, authorize_rebac]:
    print(fn.__name__, "=>", fn(req))
print("capability =>", authorize_capability(req, capability))


Each model asks a different kind of question. Real systems often combine them rather than choosing exactly one.

## 8 — Compose multiple controls

In [ ]:
def authorize_composed(request: AuthzRequest, cap: Capability) -> Decision:
    checks = [
        authorize_rbac(request),
        authorize_abac(request),
        authorize_rebac(request),
        authorize_capability(request, cap),
    ]
    failures = [d for d in checks if not d.allowed]
    if failures:
        return Decision(
            False,
            "; ".join(f"{d.model}: {d.reason}" for d in failures),
            "Composed",
        )
    return Decision(True, "RBAC + ABAC + ReBAC + capability all allow", "Composed")

print(authorize_composed(req, capability))


## 9 — PEP / PDP separation

In [ ]:
class PolicyDecisionPoint:
    def decide(self, request, capability):
        return authorize_composed(request, capability)

class RefundService:
    def __init__(self):
        self.refunds = []

    def refund(self, order_id, amount):
        result = {"order": order_id, "amount": amount, "status": "refunded"}
        self.refunds.append(result)
        return result

class ToolGateway:  # Policy Enforcement Point
    def __init__(self, pdp, service):
        self.pdp = pdp
        self.service = service

    def invoke_refund(self, request, capability):
        decision = self.pdp.decide(request, capability)
        if not decision.allowed:
            raise PermissionError(decision.reason)
        return self.service.refund(
            request.resource.removeprefix("order:"),
            request.context["amount"],
        )

gateway = ToolGateway(PolicyDecisionPoint(), RefundService())
print(gateway.invoke_refund(req, capability))


The LLM would sit **before** `ToolGateway`. It can propose the refund, but it cannot bypass the gateway's authorization decision.

## 10 — Task-scoped delegation

In [ ]:
@dataclass(frozen=True)
class Delegation:
    delegator: str
    delegate: str
    action: str
    resource: str
    max_amount: float
    task_id: str
    expires_at: datetime
    redelegation_allowed: bool = False

delegation = Delegation(
    delegator="user:alice",
    delegate="agent:refund-specialist",
    action="refund",
    resource="order:123",
    max_amount=200,
    task_id="task:928",
    expires_at=now() + timedelta(minutes=20),
)

def validate_delegation(request, d):
    if request.requester != d.delegator:
        return Decision(False, "Wrong delegator", "Delegation")
    if request.actor != d.delegate:
        return Decision(False, "Wrong delegate", "Delegation")
    if request.action != d.action or request.resource != d.resource:
        return Decision(False, "Action/resource outside delegation", "Delegation")
    if request.context.get("task_id") != d.task_id:
        return Decision(False, "Wrong task", "Delegation")
    if request.context.get("amount", float("inf")) > d.max_amount:
        return Decision(False, "Amount exceeds delegation", "Delegation")
    if now() >= d.expires_at:
        return Decision(False, "Delegation expired", "Delegation")
    return Decision(True, "Delegation valid", "Delegation")

print(validate_delegation(req, delegation))


## 11 — Prevent privilege amplification

In [ ]:
def attenuate(parent: Delegation, *, delegate, action, resource,
              max_amount, expires_at, redelegation_allowed=False):
    if not parent.redelegation_allowed:
        raise PermissionError("Parent delegation forbids re-delegation")
    if action != parent.action:
        raise PermissionError("Child action exceeds parent")
    if resource != parent.resource:
        raise PermissionError("Child resource exceeds parent")
    if max_amount > parent.max_amount:
        raise PermissionError("Child amount exceeds parent")
    if expires_at > parent.expires_at:
        raise PermissionError("Child lifetime exceeds parent")
    return Delegation(
        delegator=parent.delegate,
        delegate=delegate,
        action=action,
        resource=resource,
        max_amount=max_amount,
        task_id=parent.task_id,
        expires_at=expires_at,
        redelegation_allowed=redelegation_allowed,
    )

try:
    attenuate(
        delegation,
        delegate="agent:payments",
        action="refund",
        resource="order:123",
        max_amount=50,
        expires_at=now() + timedelta(minutes=5),
    )
except PermissionError as e:
    print("DENIED:", e)


The parent explicitly disallowed re-delegation. Security should not assume every agent may spawn equally privileged children.

## 12 — Separation of duties

In [ ]:
def approve_refund(*, creator, approver, amount):
    if creator == approver:
        return Decision(False, "Creator cannot approve own refund", "SoD")
    if amount > 1000:
        return Decision(False, "Requires human approval tier 2", "SoD")
    return Decision(True, "Independent approver accepted", "SoD")

print(approve_refund(
    creator="agent:refund-specialist",
    approver="user:manager-bob",
    amount=500,
))
print(approve_refund(
    creator="agent:refund-specialist",
    approver="agent:refund-specialist",
    amount=500,
))


## 13 — Authorization-aware retrieval

In [ ]:
DOCUMENTS = [
    {"id": "doc:north:1", "tenant": "north", "text": "North customer notes"},
    {"id": "doc:south:1", "tenant": "south", "text": "South confidential notes"},
]

def retrieve_candidates(query):
    # Simulates vector retrieval. Both documents happen to be candidates.
    return DOCUMENTS

def authorize_document(requester, doc):
    attrs = REQUESTER_ATTRIBUTES.get(requester, {})
    return attrs.get("tenant") == doc["tenant"]

def secure_retrieve(query, requester):
    candidates = retrieve_candidates(query)
    return [d for d in candidates if authorize_document(requester, d)]

print("Raw candidates :", [d["id"] for d in retrieve_candidates("customer")])
print("Model may see  :", [d["id"] for d in secure_retrieve("customer", "user:alice")])


Protected content is filtered **before** it enters model context.

## 14 — Authorization decision evidence

In [ ]:
AUDIT = []

def record_decision(request, decision, policy_version="2026.08.18.1"):
    event = {
        "timestamp": now().isoformat(),
        "requester": request.requester,
        "actor": request.actor,
        "workload": request.workload,
        "action": request.action,
        "resource": request.resource,
        "task": request.context.get("task_id"),
        "decision": "allow" if decision.allowed else "deny",
        "reason": decision.reason,
        "model": decision.model,
        "policy_version": policy_version,
    }
    AUDIT.append(event)
    return event

decision = authorize_composed(req, capability)
print(json.dumps(record_decision(req, decision), indent=2))


## 15 — Adversarial regression tests

In [ ]:
def mutate_request(**changes):
    values = {
        "requester": req.requester,
        "actor": req.actor,
        "workload": req.workload,
        "action": req.action,
        "resource": req.resource,
        "context": dict(req.context),
    }
    context_changes = changes.pop("context", None)
    values.update(changes)
    if context_changes:
        values["context"].update(context_changes)
    return AuthzRequest(**values)

tests = [
    ("valid", req, True),
    ("too much", mutate_request(context={"amount": 201}), False),
    ("wrong tenant resource", mutate_request(resource="order:999"), False),
    ("unknown action", mutate_request(action="delete_order"), False),
    ("wrong actor", mutate_request(actor="agent:research"), False),
    ("wrong task", mutate_request(context={"task_id": "task:evil"}), False),
]

for name, request, expected in tests:
    result = authorize_composed(request, capability)
    assert result.allowed is expected, (name, result)
    print(f"{name:22} {'ALLOW' if result.allowed else 'DENY'}")


## 16 — Exercise: add approval obligations

Change the system so:

```text
refund <= 200          -> agent may execute
200 < refund <= 1000   -> manager approval required
refund > 1000          -> finance approval + human confirmation
```

Instead of only `allowed: bool`, design:

```python
Decision(
    effect="conditional",
    obligations=[...]
)
```

Think about which component enforces the obligation.

## 17 — Exercise: compare policy engines

Translate the refund policy conceptually into:

### Rego

```rego
package refund.authz
default allow := false

allow if {
    input.actor == "agent:refund-specialist"
    input.action == "refund"
    input.context.amount <= 200
}
```

### Cedar

```cedar
permit (
    principal == Agent::"refund-specialist",
    action == Action::"refund",
    resource is Order
)
when {
    context.amount <= 200
};
```

### ReBAC tuples

```text
agent:refund-specialist can_act_on_behalf_of user:alice
agent:refund-specialist assigned task:928
task:928 target order:123
```

Discuss what each representation makes easiest to express.

## 18 — Architecture challenge

Design authorization for an enterprise research agent that:

- can search approved web sources;
- can read only documents the requester may read;
- can create drafts;
- cannot publish externally;
- may delegate web research to a sub-agent;
- cannot delegate internal-document access;
- must obtain human approval before sending email;
- loses all task authority after 30 minutes.

Identify:

1. PEPs;
2. PDP;
3. resource identities;
4. relationships;
5. request attributes;
6. delegation records;
7. audit evidence;
8. failure behavior if the PDP is unavailable.

## Review questions

1. Why does authentication not imply authorization?
2. What are principal/action/resource/context?
3. When does RBAC become awkward for agents?
4. What information is naturally modeled as ABAC?
5. Why does ReBAC fit many agent workflows?
6. What is capability attenuation?
7. Why must the PEP sit outside the LLM?
8. Why is task-scoped authority safer than standing privilege?
9. What does `authority(child) <= authority(parent)` prevent?
10. Why must RAG authorization happen before model exposure?
11. When should a sensitive system fail closed?
12. What evidence is needed to explain an authorization decision?

## Next course

**Beginner 05 — Least-Privilege Tool Access for Agents**
